In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import APIConnectionError, APIStatusError, APITimeoutError, OpenAI, RateLimitError
from tqdm.auto import tqdm

load_dotenv()


# ============================================================================
# 08. LLM EVALUATION ON THE SAME SUBSET, UNMASKED (FULL) TEXT
# ============================================================================
# 05_llm_evaluation.ipynb only ever showed the LLM POS-masked text.
# To measure how much the LLM depends on lexical/content cues (the
# "shortcuts" question from Uluslu & Schneider 2025), we need its accuracy
# on the SAME documents with the ORIGINAL, unmasked text, using the
# identical prompt format (forced JSON, same token budget) so masking is
# the only thing that changes between this run and 05's.
#
# IMPORTANT: this notebook does NOT resample. It reuses the exact eval_id /
# doc_id set from data/processed/results/llm_eval_subset.csv (created by 05)
# and recovers the unmasked "text" for those same rows by joining back to
# combined_nli_masked.csv.
#
# COST ESTIMATION: TOEFL11 essays are short, exam-timed (~350 words on
# average). ICLEv3 essays are untimed and can run several times longer.
# A small random dry-run sample can land disproportionately on one
# partition by chance and badly misestimate the true average cost, so this
# notebook forces the dry run to sample evenly from both partitions and
# reports token usage (and the resulting cost estimate) per partition
# instead of blending them into a single number.

BASE_DIR = Path("..").resolve()
COMBINED_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"
EVAL_SUBSET_PATH = BASE_DIR / "data/processed/results/llm_eval_subset.csv"
RESULTS_DIR = BASE_DIR / "data/processed/results"

OUTPUT_RESULTS_PATH = RESULTS_DIR / "llm_predictions_fulltext.csv"
PROGRESS_PATH = RESULTS_DIR / "llm_predictions_fulltext_progress.csv"

VALID_L1S = ["DEU", "FRA", "ITA", "JPN", "KOR", "SPA", "TUR", "ZHO"]
MODEL_NAME = "gpt-4o-mini"
MAX_COMPLETION_TOKENS = 80  # same budget as 05 -- keep the comparison fair
DRY_RUN_PER_PARTITION = 8  # calls sampled from EACH eval_partition before pausing to report cost
RANDOM_STATE = 42
RUN_FULL_EVALUATION = True  # set to False to stop after the dry-run cost estimate below
MAX_RETRIES = 5
# gpt-4o-mini pricing as of mid-2026: $0.15 / 1M input tokens, $0.60 / 1M output tokens.
# The estimate below prices every token at the input rate, so it is a rough
# lower bound; the exact figure is reported in the OpenAI usage dashboard.
PRICE_PER_MILLION_TOKENS_ESTIMATE = 0.15

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not EVAL_SUBSET_PATH.exists():
    raise FileNotFoundError(
        f"{EVAL_SUBSET_PATH} not found. Run 05_llm_evaluation.ipynb first -- "
        "this notebook reuses its exact document sample."
    )

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment.")

client = OpenAI(api_key=api_key)

SYSTEM_PROMPT = """You are an expert computational linguist evaluating Native Language Identification from English essays written by non-native speakers.

The text may contain <P> and <S> tokens marking paragraph and sentence boundaries; treat them as structural formatting markers, not content.

Choose exactly one native-language code from:
DEU, FRA, ITA, JPN, KOR, SPA, TUR, ZHO.

Return only the requested JSON object."""

RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "nli_prediction",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "predicted_l1": {"type": "string", "enum": VALID_L1S},
                "confidence_score": {"type": "number", "minimum": 0, "maximum": 1},
            },
            "required": ["predicted_l1", "confidence_score"],
            "additionalProperties": False,
        },
    },
}


def load_eval_subset_with_fulltext() -> pd.DataFrame:
    eval_subset = pd.read_csv(EVAL_SUBSET_PATH)
    combined = pd.read_csv(COMBINED_CSV_PATH)[["doc_id", "text"]]
    combined["text"] = combined["text"].fillna("").astype(str)

    merged = eval_subset.merge(combined, on="doc_id", how="left", validate="one_to_one")
    missing_text = merged["text"].isna().sum()
    if missing_text:
        raise ValueError(f"{missing_text} rows in the eval subset have no matching unmasked text.")

    return merged


def build_processing_order(df: pd.DataFrame, dry_run_per_partition: int, random_state: int) -> pd.DataFrame:
    """Reorder rows so the dry-run sample draws evenly from every eval_partition
    (TOEFL11, ICLE), instead of relying on whatever partition happens to appear
    first in the already-shuffled row order. Without this, a small dry run could
    by chance include mostly short TOEFL11 essays or mostly long ICLE essays and
    badly misestimate the true average cost."""
    dry_run_parts, remainder_parts = [], []
    for _, part_df in df.groupby("eval_partition", sort=False):
        dry_run_parts.append(part_df.head(dry_run_per_partition))
        remainder_parts.append(part_df.iloc[dry_run_per_partition:])

    dry_run_rows = pd.concat(dry_run_parts).sample(frac=1, random_state=random_state)
    remainder_rows = pd.concat(remainder_parts).sample(frac=1, random_state=random_state)
    return pd.concat([dry_run_rows, remainder_rows], ignore_index=False)


def summarize_dry_run(progress_df: pd.DataFrame, eval_df: pd.DataFrame, price_per_million_tokens: float) -> None:
    """Print a per-partition token/cost estimate instead of one blended mean.
    See the notebook header for why blending TOEFL11 and ICLE token counts
    together can be misleading."""
    dry_run_df = progress_df[progress_df["total_tokens"].notna()]
    partition_sizes = eval_df["eval_partition"].value_counts()

    print("Dry run complete. Token usage by partition:")
    weighted_total_tokens = 0.0
    for partition, part_dry in dry_run_df.groupby("eval_partition"):
        mean_tokens = part_dry["total_tokens"].mean()
        n_full = int(partition_sizes.get(partition, 0))
        partition_estimate = mean_tokens * n_full
        weighted_total_tokens += partition_estimate
        print(
            f"  {partition}: mean {mean_tokens:.1f} tokens/call "
            f"(from {len(part_dry)} sampled calls) -> ~{partition_estimate:.0f} tokens across all {n_full} rows"
        )

    missing_partitions = set(partition_sizes.index) - set(dry_run_df["eval_partition"].unique())
    if missing_partitions:
        print(f"  WARNING: no dry-run calls yet for {sorted(missing_partitions)} -- estimate below is incomplete.")

    estimated_cost_usd = weighted_total_tokens / 1_000_000 * price_per_million_tokens
    print(f"Weighted total estimated tokens (all partitions): {weighted_total_tokens:.0f}")
    print(f"Rough estimated cost: ${estimated_cost_usd:.3f}")
    print("Inspect cost manually (and your OpenAI usage dashboard) before continuing.")


def call_llm(text: str) -> dict:
    user_prompt = f"Essay:\n\n{text}"

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                response_format=RESPONSE_FORMAT,
                temperature=0,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )

            raw_content = response.choices[0].message.content
            parsed = json.loads(raw_content)

            usage = response.usage
            return {
                "predicted_l1": parsed["predicted_l1"],
                "confidence_score": parsed["confidence_score"],
                "raw_response": raw_content,
                "prompt_tokens": usage.prompt_tokens if usage else None,
                "completion_tokens": usage.completion_tokens if usage else None,
                "total_tokens": usage.total_tokens if usage else None,
                "error": "",
            }

        except (RateLimitError, APITimeoutError, APIConnectionError, APIStatusError) as e:
            wait_seconds = min(60, 2**attempt)
            if attempt == MAX_RETRIES - 1:
                return {
                    "predicted_l1": None,
                    "confidence_score": None,
                    "raw_response": "",
                    "prompt_tokens": None,
                    "completion_tokens": None,
                    "total_tokens": None,
                    "error": repr(e),
                }
            time.sleep(wait_seconds)


eval_df = load_eval_subset_with_fulltext()
eval_df = build_processing_order(eval_df, DRY_RUN_PER_PARTITION, RANDOM_STATE)
n_partitions = eval_df["eval_partition"].nunique()
DRY_RUN_N = DRY_RUN_PER_PARTITION * n_partitions

print(f"Reusing {len(eval_df)} rows from the 05 evaluation subset (unmasked text).")
print(f"Dry run will sample {DRY_RUN_PER_PARTITION} calls from each of {n_partitions} partitions ({DRY_RUN_N} total).")
print(eval_df.groupby(["eval_partition", "label"]).size().unstack(fill_value=0))

existing = pd.DataFrame()
completed_ids = set()
if PROGRESS_PATH.exists():
    existing = pd.read_csv(PROGRESS_PATH)
    completed_ids = set(existing["eval_id"])
    print(f"Resuming from progress file. Completed rows: {len(completed_ids)}")

rows = [] if existing.empty else existing.to_dict("records")
new_calls_made = 0

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    if row["eval_id"] in completed_ids:
        continue

    result = call_llm(row["text"])
    output_row = {
        "eval_id": row["eval_id"],
        "doc_id": row["doc_id"],
        "dataset": row["dataset"],
        "split": row["split"],
        "eval_partition": row["eval_partition"],
        "true_l1": row["label"],
        "model": MODEL_NAME,
        "text_condition": "fulltext",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        **result,
    }
    rows.append(output_row)
    new_calls_made += 1

    pd.DataFrame(rows).to_csv(PROGRESS_PATH, index=False)

    if new_calls_made == DRY_RUN_N:
        progress_df = pd.DataFrame(rows)
        summarize_dry_run(progress_df, eval_df, PRICE_PER_MILLION_TOKENS_ESTIMATE)
        print("To run all remaining rows, set RUN_FULL_EVALUATION = True and rerun this notebook.")
        if not RUN_FULL_EVALUATION:
            break

results_df = pd.DataFrame(rows)
results_df.to_csv(OUTPUT_RESULTS_PATH, index=False)

print(f"Saved LLM full-text predictions: {OUTPUT_RESULTS_PATH}")
print(results_df["error"].replace("", pd.NA).dropna().head())
